In [ ]:
import pandas as pd

import pyarrow as pa

In [1]:
import importlib
import pathlib
import sys
from datetime import date

import pyarrow as pa


# 按项目规范定位根目录
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

PROJECT_ROOT = candidate_root
PROJECT_ROOT

WindowsPath('E:/Latitude_Analytics_v2')

In [6]:
# 02_Quant_Trading 以数字开头，使用 importlib 导入
lakehouse = importlib.import_module("02_Quant_Trading.a01_Data_Collection.c00_lakehouse")

from config.data_contracts import FUTURES_DAILY_SCHEMA


lake_root = (
    PROJECT_ROOT
    / "03_Futures_Database"
    / "futures_lake"
    / "silver"
)

daily_path = lake_root / "fact_futures_daily"

daily_partitioning = lakehouse.hive_partitioning(
    [
        pa.field("exchange_code", pa.string()),
        pa.field("year", pa.int16()),
        pa.field("month", pa.int8()),
    ]
)

# 只读取需要的列，减少内存占用
daily_df = lakehouse.read_dataset(
    daily_path,
    daily_partitioning,
    FUTURES_DAILY_SCHEMA,
    [
        "contract_code",
        "exchange_code",
        "underlying_code",
        "trading_date",
        "close",
        "volume",
        "has_market_data",
        "year",
        "month",
    ],
)
daily_df

,contract_code,exchange_code,underlying_code,trading_date,close,volume,has_market_data,year,month
0,LC2401.GFEX,GFEX,LC,2024-01-02,102450.0,593.0,True,2024,1
1,LC2402.GFEX,GFEX,LC,2024-01-02,102850.0,2670.0,True,2024,1
2,LC2403.GFEX,GFEX,LC,2024-01-02,103500.0,8237.0,True,2024,1
3,LC2404.GFEX,GFEX,LC,2024-01-02,107650.0,6426.0,True,2024,1
4,LC2405.GFEX,GFEX,LC,2024-01-02,107950.0,6217.0,True,2024,1
...,...,...,...,...,...,...,...,...,...
478604,ZC2703.XZCE,XZCE,ZC,2026-07-24,801.4,0.0,True,2026,7
478605,ZC2704.XZCE,XZCE,ZC,2026-07-24,801.4,0.0,True,2026,7
478606,ZC2705.XZCE,XZCE,ZC,2026-07-24,801.4,0.0,True,2026,7
478607,ZC2706.XZCE,XZCE,ZC,2026-07-24,801.4,0.0,True,2026,7


In [5]:
# 筛选 2024 年 1 月的沪铜
cu_df = daily_df[
    (daily_df["underlying_code"] == "CU")
    & (daily_df["trading_date"] >= date(2024, 1, 1))
    & (daily_df["trading_date"] <= date(2024, 1, 31))
    & daily_df["has_market_data"]
].sort_values(["trading_date", "contract_code"])

cu_df

,contract_code,exchange_code,underlying_code,trading_date,close,volume,has_market_data,year,month
196739,CU2401.XSGE,XSGE,CU,2024-01-02,69030.0,13614.0,True,2024,1
196740,CU2402.XSGE,XSGE,CU,2024-01-02,68960.0,31322.0,True,2024,1
196741,CU2403.XSGE,XSGE,CU,2024-01-02,68940.0,19205.0,True,2024,1
196742,CU2404.XSGE,XSGE,CU,2024-01-02,68910.0,5307.0,True,2024,1
196743,CU2405.XSGE,XSGE,CU,2024-01-02,68910.0,1576.0,True,2024,1
...,...,...,...,...,...,...,...,...,...
201240,CU2409.XSGE,XSGE,CU,2024-01-31,68930.0,645.0,True,2024,1
201241,CU2410.XSGE,XSGE,CU,2024-01-31,68830.0,138.0,True,2024,1
201242,CU2411.XSGE,XSGE,CU,2024-01-31,68790.0,24.0,True,2024,1
201243,CU2412.XSGE,XSGE,CU,2024-01-31,68670.0,97.0,True,2024,1
